# 03 - Gold: Tabelas de Negócio para Dashboard

Camada Gold do projeto Olist E-commerce. Cada tabela aqui alimenta um painel específico
do dashboard "Visão Geral do Negócio":

1. `receita_mensal` → Evolução de receita
2. `receita_por_estado` → Onde estão nossos clientes
3. `performance_entrega_mensal` → Performance de entrega
4. `satisfacao_entrega` → Satisfação do cliente x atraso
5. `ranking_vendedores` → Top vendedores

In [0]:
#Setup (imports + carregamento das tabelas Silver) 
from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime

CATALOG = "olist_project"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

orders = spark.table(f"{SILVER_SCHEMA}.orders")
customers = spark.table(f"{SILVER_SCHEMA}.customers")
order_items = spark.table(f"{SILVER_SCHEMA}.order_items")
payments = spark.table(f"{SILVER_SCHEMA}.payments")
reviews = spark.table(f"{SILVER_SCHEMA}.reviews")
products = spark.table(f"{SILVER_SCHEMA}.products")
sellers = spark.table(f"{SILVER_SCHEMA}.sellers")
geolocation = spark.table(f"{SILVER_SCHEMA}.geolocation")
category_translation = spark.table(f"{SILVER_SCHEMA}.category_translation")

print("✅ Setup concluído")

In [0]:
#Função de log de qualidade (mesmo padrão da Silver)
def registrar_checagem(camada, tabela, checagem, status, detalhe=""):
    row = Row(camada=camada, tabela=tabela, checagem=checagem,
              status=status, detalhe=detalhe, timestamp=datetime.now())
    spark.createDataFrame([row]).write.format("delta").mode("append") \
        .saveAsTable(f"{GOLD_SCHEMA}.quality_log")
    simbolo = "✅" if status == "PASSOU" else "❌"
    print(f"{simbolo} [{camada}.{tabela}] {checagem} — {detalhe}")

## 1. Receita Mensal
**Painel:** Evolução de Receita
**Pergunta:** Estamos crescendo mês a mês?
**Grão:** um mês

In [0]:
#Construçao
receita_mensal = (orders.join(order_items, "order_id")
    .withColumn("mes", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .groupBy("mes")
    .agg(
        F.sum("price").alias("receita_produtos"),
        F.sum("freight_value").alias("receita_frete"),
        F.countDistinct("order_id").alias("qtd_pedidos")
    )
    .orderBy("mes")
)

# Variação % em relação ao mês anterior
from pyspark.sql.window import Window

janela_mensal = Window.orderBy("mes")

receita_mensal = receita_mensal.withColumn(
    "receita_mes_anterior", F.lag("receita_produtos").over(janela_mensal)
).withColumn(
    "variacao_pct",
    F.round(
        (F.col("receita_produtos") - F.col("receita_mes_anterior")) / F.col("receita_mes_anterior") * 100,
        2
    )
).drop("receita_mes_anterior")

display(receita_mensal)

In [0]:
#Gravação
receita_mensal.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_SCHEMA}.receita_mensal")

print(f"✅ gold.receita_mensal: {receita_mensal.count()} linhas")

In [0]:
#Validação
# Reconciliação: soma da Gold deve bater com a soma direta da Silver
receita_gold = spark.table(f"{GOLD_SCHEMA}.receita_mensal").agg(F.sum("receita_produtos")).collect()[0][0]
receita_silver = order_items.agg(F.sum("price")).collect()[0][0]
diferenca = abs(float(receita_gold) - float(receita_silver))

status = "PASSOU" if diferenca < 0.01 else "FALHOU"
registrar_checagem("gold", "receita_mensal", "reconciliacao_com_silver", status,
                    f"gold={receita_gold}, silver={receita_silver}")

# Sem valores negativos
negativos = spark.table(f"{GOLD_SCHEMA}.receita_mensal").filter(F.col("receita_produtos") < 0).count()
status = "PASSOU" if negativos == 0 else "FALHOU"
registrar_checagem("gold", "receita_mensal", "sem_valores_negativos", status, f"{negativos} negativos")

## 2. Receita por Estado
**Painel:** Onde estão nossos clientes
**Pergunta:** Quais estados compram mais / têm maior ticket médio?
**Grão:** um estado

In [0]:
#Construção
receita_por_estado = (orders.join(order_items, "order_id")
    .join(customers, "customer_id")
    .groupBy(F.col("customer_state").alias("estado"))
    .agg(
        F.sum("price").alias("receita_total"),
        F.countDistinct("order_id").alias("qtd_pedidos"),
        F.round(F.avg("price"), 2).alias("ticket_medio")
    )
    .orderBy(F.desc("receita_total"))
)

display(receita_por_estado)

In [0]:
#Gravação
receita_por_estado.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_SCHEMA}.receita_por_estado")

print(f"✅ gold.receita_por_estado: {receita_por_estado.count()} linhas")

In [0]:
#Validação
total_gold = spark.table(f"{GOLD_SCHEMA}.receita_por_estado").agg(F.sum("receita_total")).collect()[0][0]
diferenca = abs(float(total_gold) - float(receita_silver))
status = "PASSOU" if diferenca < 0.01 else "FALHOU"
registrar_checagem("gold", "receita_por_estado", "reconciliacao_com_silver", status,
                    f"gold={total_gold}, silver={receita_silver}")

estados_nulos = spark.table(f"{GOLD_SCHEMA}.receita_por_estado").filter(F.col("estado").isNull()).count()
status = "PASSOU" if estados_nulos == 0 else "FALHOU"
registrar_checagem("gold", "receita_por_estado", "sem_estado_nulo", status, f"{estados_nulos} nulos")

## 3. Performance de Entrega Mensal
**Painel:** Performance de Entrega
**Pergunta:** Estamos entregando no prazo prometido?
**Grão:** um mês
**Regra de negócio:** só considera pedidos com status 'delivered' e data de entrega preenchida.

In [0]:
#Construção
pedidos_entregues = (orders
    .filter((F.col("order_status") == "delivered") &
            F.col("order_delivered_customer_date").isNotNull() &
            (F.col("flag_data_inconsistente") == False))  # exclui os que já sinalizamos como inconsistentes
    .withColumn("mes", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .withColumn("dias_entrega",
        F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    .withColumn("no_prazo",
        F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date"))
)

performance_entrega_mensal = (pedidos_entregues
    .groupBy("mes")
    .agg(
        F.round(F.avg("dias_entrega"), 1).alias("tempo_medio_entrega_dias"),
        F.round(F.avg(F.col("no_prazo").cast("int")) * 100, 2).alias("pct_entregas_no_prazo"),
        F.countDistinct("order_id").alias("qtd_pedidos_entregues")
    )
    .withColumn("pct_entregas_atrasadas", F.round(100 - F.col("pct_entregas_no_prazo"), 2))
    .orderBy("mes")
)

display(performance_entrega_mensal)

In [0]:
#Gravação
performance_entrega_mensal.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_SCHEMA}.performance_entrega_mensal")

print(f"✅ gold.performance_entrega_mensal: {performance_entrega_mensal.count()} linhas")

In [0]:
#Validação
# percentuais devem estar entre 0 e 100
df_check = spark.table(f"{GOLD_SCHEMA}.performance_entrega_mensal")
fora_intervalo = df_check.filter(
    (F.col("pct_entregas_no_prazo") < 0) | (F.col("pct_entregas_no_prazo") > 100)
).count()
status = "PASSOU" if fora_intervalo == 0 else "FALHOU"
registrar_checagem("gold", "performance_entrega_mensal", "percentual_no_intervalo_0_100", status,
                    f"{fora_intervalo} fora do intervalo")

# tempo médio de entrega não pode ser negativo
negativos = df_check.filter(F.col("tempo_medio_entrega_dias") < 0).count()
status = "PASSOU" if negativos == 0 else "FALHOU"
registrar_checagem("gold", "performance_entrega_mensal", "tempo_entrega_nao_negativo", status,
                    f"{negativos} negativos")

## 4. Satisfação x Entrega
**Painel:** Satisfação do Cliente
**Pergunta:** Atraso na entrega afeta a nota do cliente?
**Grão:** status de entrega (no_prazo / atrasado)

In [0]:
#Construção
satisfacao_entrega = (pedidos_entregues.join(reviews, "order_id")
    .withColumn("status_entrega", F.when(F.col("no_prazo"), "no_prazo").otherwise("atrasado"))
    .groupBy("status_entrega")
    .agg(
        F.round(F.avg("review_score"), 2).alias("nota_media"),
        F.count("review_id").alias("qtd_avaliacoes")
    )
    .orderBy("status_entrega")
)

display(satisfacao_entrega)

In [0]:
#Gravação
satisfacao_entrega.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_SCHEMA}.satisfacao_entrega")

print(f"✅ gold.satisfacao_entrega: {satisfacao_entrega.count()} linhas")

In [0]:
#Validação
df_check = spark.table(f"{GOLD_SCHEMA}.satisfacao_entrega")

# nota média deve estar entre 1 e 5
fora_intervalo = df_check.filter((F.col("nota_media") < 1) | (F.col("nota_media") > 5)).count()
status = "PASSOU" if fora_intervalo == 0 else "FALHOU"
registrar_checagem("gold", "satisfacao_entrega", "nota_media_no_intervalo_1_5", status,
                    f"{fora_intervalo} fora do intervalo")

# devem existir exatamente as 2 categorias esperadas
categorias = [row["status_entrega"] for row in df_check.select("status_entrega").collect()]
status = "PASSOU" if set(categorias) == {"no_prazo", "atrasado"} else "FALHOU"
registrar_checagem("gold", "satisfacao_entrega", "categorias_esperadas", status, f"{categorias}")

## 5. Ranking de Vendedores
**Painel:** Top Vendedores
**Pergunta:** Quem são nossos melhores vendedores (volume + qualidade)?
**Grão:** um vendedor
**Regra de negócio:** só considera vendedores com pelo menos 5 pedidos, pra evitar distorção de nota com poucos dados.

In [0]:
#Construção
ranking_vendedores = (order_items.join(orders, "order_id")
    .join(sellers, "seller_id")
    .join(reviews, "order_id", "left")  # left join: nem todo pedido tem review
    .groupBy("seller_id", "seller_state")
    .agg(
        F.sum("price").alias("receita_total"),
        F.countDistinct(F.col("order_id")).alias("qtd_pedidos"),
        F.round(F.avg("review_score"), 2).alias("nota_media")
    )
    .filter(F.col("qtd_pedidos") >= 5)
    .orderBy(F.desc("receita_total"))
)

display(ranking_vendedores)

In [0]:
#Gravação
ranking_vendedores.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_SCHEMA}.ranking_vendedores")

print(f"✅ gold.ranking_vendedores: {ranking_vendedores.count()} linhas")

In [0]:
#Validação
df_check = spark.table(f"{GOLD_SCHEMA}.ranking_vendedores")

# seller_id deve ser único (grão = um vendedor)
total = df_check.count()
distintos = df_check.select("seller_id").distinct().count()
status = "PASSOU" if total == distintos else "FALHOU"
registrar_checagem("gold", "ranking_vendedores", "chave_unica_seller_id", status, f"{total} vs {distintos}")

# regra de negócio: filtro de >=5 pedidos foi respeitado
abaixo_minimo = df_check.filter(F.col("qtd_pedidos") < 5).count()
status = "PASSOU" if abaixo_minimo == 0 else "FALHOU"
registrar_checagem("gold", "ranking_vendedores", "filtro_minimo_pedidos_respeitado", status,
                    f"{abaixo_minimo} abaixo do mínimo")

## Resumo de Validação — Camada Gold

In [0]:
display(
    spark.table(f"{GOLD_SCHEMA}.quality_log")
    .filter("camada = 'gold'")
    .groupBy("tabela", "status")
    .count()
    .orderBy("tabela")
)